# All-vs-all input correlation screen

Public project: [CastaliaInstitute/solar-revolutions](https://github.com/CastaliaInstitute/solar-revolutions)  
Public page: [castaliainstitute.github.io/solar-revolutions](https://castaliainstitute.github.io/solar-revolutions/)

## Question

Which of our catalog inputs correlate with each other?

## Scientific posture

This is a discovery screen, not inference. With many series and many pairwise tests, strong-looking correlations are expected by chance, shared trends, short overlaps, and autocorrelation. Treat results as candidates for later preregistered tests.


## 1. Methods

- Annualize every available input series.
- Compute pairwise same-year Pearson and Spearman correlations.
- Require a minimum overlap before ranking pairs.
- Report raw ranks, module-pair summaries, and high-correlation clusters.
- Run a lag scan for the top same-year pairs.
- Run circular-shift placebo checks for the top pairs to identify correlations that may be time-structure artifacts.


## Current executed readout

Latest local run loaded `93` usable annual input series and computed `4,030` same-year pairs with at least 20 overlapping years. The raw top pairs are dominated by duplicate or near-duplicate measurements: population vs WDI population, mobile subscriptions vs WDI mobile subscriptions, internet users vs WDI internet users, global inflation proxy vs WDI inflation, and related democracy-index families.

Current top raw pair: `demographics.population` vs `demographics.wdi_population`, `r=1.000`, `n=66`. This is a data-duplication/same-construct result, not a discovery. The notebook therefore includes a deduplicated cross-module view for more useful browsing.

Circular-shift placebo is passed by many top raw pairs because many are duplicate measurements or strong monotonic trends. Passing shift placebo is not enough; mechanism review and out-of-sample validation are still required.


## 2. Setup


In [ ]:
#@title Install notebook dependencies { display-mode: "form" }
INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2,<3", "numpy>=1.26,<3", "scipy>=1.11,<2", "networkx>=3.2,<4"], check=True)


In [ ]:
#@title Locate or clone public repository { display-mode: "form" }
import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/CastaliaInstitute/solar-revolutions.git"

def running_in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def api_root_for(candidate: Path):
    for api in [candidate / "api", candidate / "dashboard" / "public" / "api"]:
        if (api / "input-series.json").exists() and (api / "input-data").exists():
            return api
    return None

def find_repo():
    env_root = os.environ.get("SOLAR_REPO_ROOT") or os.environ.get("APOCALYPSO_REPO_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/solar-revolutions")])
    for candidate in candidates:
        api = api_root_for(candidate)
        if api is not None:
            return candidate, api
    if running_in_colab():
        target = Path("/content/solar-revolutions")
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
        api = api_root_for(target)
        if api is not None:
            return target, api
    raise FileNotFoundError("Could not find public API artifacts. Set SOLAR_REPO_ROOT to the solar-revolutions checkout.")

ROOT, API = find_repo()
INPUT_DATA = API / "input-data"
print("Repo root:", ROOT)
print("API:", API)


## 3. Load and annualize input series


In [ ]:
#@title Load annualized series { display-mode: "form" }
MIN_OVERLAP = 20

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def parse_year(value):
    try:
        return int(str(value)[:4])
    except Exception:
        return None

def series_path(series_id):
    return INPUT_DATA / f"{series_id.replace('.', '_')}.json"

def annualize(series_id):
    data = load_json(series_path(series_id))
    buckets = {}
    for point in data.get("points", []):
        year = point.get("year") if isinstance(point.get("year"), int) else parse_year(point.get("date"))
        try:
            value = float(point.get("value"))
        except Exception:
            continue
        if year is None or not np.isfinite(value):
            continue
        buckets.setdefault(year, []).append(value)
    return pd.Series({year: float(np.mean(values)) for year, values in buckets.items()}, name=series_id).sort_index()

catalog = load_json(API / "input-series.json")["series"]
catalog_by_id = {item["id"]: item for item in catalog}
available_ids = [item["id"] for item in catalog if series_path(item["id"]).exists()]
series = {}
for sid in available_ids:
    s = annualize(sid)
    if len(s.dropna()) >= MIN_OVERLAP and s.dropna().std() > 0:
        series[sid] = s
print("Loaded series:", len(series))
pd.DataFrame([{
    "id": sid,
    "name": catalog_by_id.get(sid, {}).get("name", sid),
    "module": catalog_by_id.get(sid, {}).get("module_id", "unknown"),
    "n": len(s.dropna()),
    "start": int(s.dropna().index.min()),
    "end": int(s.dropna().index.max()),
} for sid, s in series.items()]).sort_values("id").head(20)


## 4. Same-year all-vs-all correlations


In [ ]:
#@title Compute all pairwise same-year correlations { display-mode: "form" }
def pearson(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3 or x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def spearman(x, y):
    return pearson(pd.Series(x).rank().to_numpy(), pd.Series(y).rank().to_numpy())

ids = list(series.keys())
rows = []
for i, a_id in enumerate(ids):
    a = series[a_id]
    for b_id in ids[i + 1:]:
        b = series[b_id]
        years = sorted(set(a.index).intersection(b.index))
        if len(years) < MIN_OVERLAP:
            continue
        x = [a.loc[year] for year in years]
        y = [b.loc[year] for year in years]
        r = pearson(x, y)
        rho = spearman(x, y)
        rows.append({
            "a_id": a_id,
            "a_name": catalog_by_id.get(a_id, {}).get("name", a_id),
            "a_module": catalog_by_id.get(a_id, {}).get("module_id", "unknown"),
            "b_id": b_id,
            "b_name": catalog_by_id.get(b_id, {}).get("name", b_id),
            "b_module": catalog_by_id.get(b_id, {}).get("module_id", "unknown"),
            "n": len(years),
            "start": years[0],
            "end": years[-1],
            "pearson_r": r,
            "spearman_rho": rho,
            "abs_pearson": abs(r) if np.isfinite(r) else np.nan,
            "abs_spearman": abs(rho) if np.isfinite(rho) else np.nan,
        })
all_pairs = pd.DataFrame(rows).sort_values("abs_pearson", ascending=False)
print("Pair count:", len(all_pairs))
all_pairs.head(30)


## 5. Deduplicated / cross-module view

The raw top table is dominated by duplicate measurements, levels, and broad trends. This view removes same-module pairs and obvious duplicate namespaces so the browsing surface is more useful.


In [ ]:
#@title Remove obvious duplicate/same-module pairs { display-mode: "form" }
def namespace(series_id):
    return series_id.split(".", 1)[0]

deduped_pairs = all_pairs.copy()
deduped_pairs["a_namespace"] = deduped_pairs["a_id"].map(namespace)
deduped_pairs["b_namespace"] = deduped_pairs["b_id"].map(namespace)
deduped_pairs = deduped_pairs[deduped_pairs["a_module"] != deduped_pairs["b_module"]]
deduped_pairs = deduped_pairs[deduped_pairs["a_namespace"] != deduped_pairs["b_namespace"]]
deduped_pairs = deduped_pairs[~deduped_pairs["a_id"].str.contains("wdi_") | ~deduped_pairs["b_id"].str.contains("wdi_")]
deduped_pairs.sort_values("abs_pearson", ascending=False).head(50)


## 5. Browse filters

Use these cells to browse without rerunning the whole screen.


In [ ]:
#@title Browse pair rankings { display-mode: "form" }
MIN_N = 30  #@param {type:"integer"}
EXCLUDE_SAME_MODULE = False  #@param {type:"boolean"}
QUERY = ""  #@param {type:"string"}
browse = all_pairs[all_pairs["n"] >= MIN_N].copy()
if EXCLUDE_SAME_MODULE:
    browse = browse[browse["a_module"] != browse["b_module"]]
if QUERY.strip():
    q = QUERY.lower().strip()
    browse = browse[browse.apply(lambda row: q in f"{row.a_id} {row.a_name} {row.b_id} {row.b_name}".lower(), axis=1)]
browse.sort_values("abs_pearson", ascending=False).head(50)


## 6. Module-pair summary


In [ ]:
#@title Summarize correlations by module pair { display-mode: "form" }
module_rows = []
for _, row in all_pairs.iterrows():
    modules = sorted([row.a_module, row.b_module])
    module_rows.append({"module_pair": " / ".join(modules), "abs_pearson": row.abs_pearson, "n": row.n})
module_summary = pd.DataFrame(module_rows).groupby("module_pair").agg(
    pairs=("abs_pearson", "count"),
    median_abs_r=("abs_pearson", "median"),
    p90_abs_r=("abs_pearson", lambda x: float(np.nanpercentile(x, 90))),
    max_abs_r=("abs_pearson", "max"),
    median_overlap=("n", "median"),
).reset_index().sort_values("max_abs_r", ascending=False)
module_summary.head(40)


## 7. Lag scan for top same-year pairs

This is expensive-ish but limited to the top raw pairs. Use it for candidate generation only.


In [ ]:
#@title Lag scan top pairs { display-mode: "form" }
TOP_N = 100  #@param {type:"integer"}
MAX_LAG = 11  #@param {type:"integer"}
lag_rows = []
for _, row in all_pairs.head(TOP_N).iterrows():
    a = series[row.a_id]
    b = series[row.b_id]
    best = None
    for lag in range(-MAX_LAG, MAX_LAG + 1):
        shifted = a.copy()
        shifted.index = shifted.index + lag
        years = sorted(set(shifted.index).intersection(b.index))
        if len(years) < MIN_OVERLAP:
            continue
        r = pearson([shifted.loc[year] for year in years], [b.loc[year] for year in years])
        candidate = {"a_id": row.a_id, "b_id": row.b_id, "lag": lag, "n": len(years), "pearson_r": r, "abs_pearson": abs(r) if np.isfinite(r) else np.nan}
        if best is None or candidate["abs_pearson"] > best["abs_pearson"]:
            best = candidate
    if best is not None:
        best.update({"a_name": row.a_name, "b_name": row.b_name, "same_year_abs_r": row.abs_pearson})
        lag_rows.append(best)
lag_results = pd.DataFrame(lag_rows).sort_values("abs_pearson", ascending=False)
lag_results.head(50)


## 8. Circular-shift placebo checks

For the strongest same-year pairs, compare observed correlation to circular shifts of the first series. This catches many cyclic/time-structure artifacts.


In [ ]:
#@title Circular-shift placebo for top pairs { display-mode: "form" }
PLACEBO_TOP_N = 50  #@param {type:"integer"}
placebo_rows = []
for _, row in all_pairs.head(PLACEBO_TOP_N).iterrows():
    a = series[row.a_id]
    b = series[row.b_id]
    years = sorted(set(a.index).intersection(b.index))
    x = np.asarray([a.loc[year] for year in years], dtype=float)
    y = np.asarray([b.loc[year] for year in years], dtype=float)
    observed = pearson(x, y)
    shifted = []
    for shift in range(1, len(x)):
        shifted.append(abs(pearson(np.roll(x, shift), y)))
    p_two_sided = float(np.mean(np.asarray(shifted) >= abs(observed))) if shifted else np.nan
    placebo_rows.append({
        "a_id": row.a_id,
        "a_name": row.a_name,
        "b_id": row.b_id,
        "b_name": row.b_name,
        "n": int(row.n),
        "observed_r": observed,
        "abs_observed_r": abs(observed),
        "circular_shift_p_two_sided": p_two_sided,
        "beats_shift_p05": p_two_sided < 0.05,
    })
placebo_results = pd.DataFrame(placebo_rows).sort_values(["beats_shift_p05", "abs_observed_r"], ascending=[False, False])
placebo_results


## 9. Cycle-prone / correlation-prone inputs

Inputs that appear repeatedly in top correlations may be broad trend/cycle markers rather than specific mechanisms.


In [ ]:
#@title Inputs appearing most often in top correlations { display-mode: "form" }
TOP_FOR_COUNTS = 250  #@param {type:"integer"}
counts = {}
for _, row in all_pairs.head(TOP_FOR_COUNTS).iterrows():
    for side in ["a", "b"]:
        sid = row[f"{side}_id"]
        counts.setdefault(sid, {"id": sid, "name": catalog_by_id.get(sid, {}).get("name", sid), "module": catalog_by_id.get(sid, {}).get("module_id", "unknown"), "top_pair_count": 0})
        counts[sid]["top_pair_count"] += 1
correlation_prone = pd.DataFrame(counts.values()).sort_values("top_pair_count", ascending=False)
correlation_prone.head(40)


## 10. Computed skeptical verdict


In [ ]:
#@title Computed skeptical verdict { display-mode: "form" }
top = all_pairs.iloc[0] if len(all_pairs) else None
placebo_pass = int(placebo_results["beats_shift_p05"].sum()) if len(placebo_results) else 0
top_text = "none" if top is None else f"{top.a_id} vs {top.b_id}: r={top.pearson_r:.3f}, n={int(top.n)}"
print("Loaded series:", len(series))
print("Pair count:", len(all_pairs))
print("Top same-year pair:", top_text)
print(f"Top {PLACEBO_TOP_N} pairs beating circular-shift p<0.05:", placebo_pass)
if placebo_pass == 0:
    print("Verdict: top all-vs-all correlations do not beat circular-shift placebo; treat as trend/cycle artifacts.")
elif placebo_pass < PLACEBO_TOP_N / 4:
    print("Verdict: some top pairs beat circular-shift placebo, but most should remain exploratory candidates under multiple-testing pressure.")
else:
    print("Verdict: many top pairs beat circular-shift placebo; prioritize mechanism review and out-of-sample validation.")


## 11. Next steps

Use this notebook to pick candidate pairs. Do not interpret any pair causally until it survives a preregistered model, out-of-sample validation, conventional controls, and domain-specific mechanism review.
